# Multi-Process Service (MPS)

A practical reference for NVIDIA's CUDA Multi-Process Service (MPS) and how it lets
several processes share a single GPU concurrently in AI/ML and HPC workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

The **Multi-Process Service (MPS)** is a binary-compatible client/server
implementation of the CUDA Application Programming Interface. It lets *multiple
CUDA processes* run their kernels concurrently on a **single GPU** by funnelling
their work through one shared GPU context, instead of having the GPU time-slice
between a separate context per process.

### What is it?

A normal CUDA process owns its own GPU context. When several processes target the
same GPU, the driver **time-slices**: it serially swaps one process's context onto
the GPU, runs its kernels, then swaps the next one in. Only one process executes at
a time, and small kernels from one process leave most of the Streaming
Multiprocessors (SMs) idle.

MPS replaces that model with a daemon-managed **server process** that owns a single
CUDA context. Client processes connect to the server over a named pipe; their kernel
launches and memory copies are interleaved into the shared context, so kernels from
*different processes* can occupy the GPU **at the same time** (spatial sharing). MPS
is fully transparent — applications need no code changes.

### Why use it?

- **Higher GPU utilization** — fills idle SMs by running kernels from many processes
  concurrently, ideal when no single process saturates the GPU.
- **Lower context-switch overhead** — one context for all clients eliminates the
  expensive driver context swaps of pure time-slicing.
- **Reduced launch latency / better overlap** — kernels and copies from different
  clients overlap rather than queuing behind a context switch.
- **Resource provisioning** — on Volta and newer you can cap each client's share of
  SMs (`CUDA_MPS_ACTIVE_THREAD_PERCENTAGE`) and pinned device memory.
- **No application changes** — purely a runtime/launch-time concern.

### When to use it?

- Many **small or bursty** processes that each under-use the GPU (e.g. several model
  replicas, or per-rank MPI processes).
- **Inference serving** where you want to pack multiple lightweight models onto one
  GPU for throughput.
- **HPC / MPI** jobs that historically assigned one rank per GPU and left it idle.
- Consolidating several **independent jobs of the same trusted user** onto one GPU to
  raise throughput, where strong hardware isolation (MIG) is not required.

## Key Features

### Core Capabilities of MPS

| Feature | Description | Benefit |
|---------|-------------|---------|
| Shared GPU context | All clients submit work through one server context | True concurrent kernel execution across processes |
| Transparent client runtime | `libcuda` routes through the MPS server automatically | Zero application code changes |
| Active-thread provisioning | `CUDA_MPS_ACTIVE_THREAD_PERCENTAGE` limits a client's SM share | Predictable partitioning / QoS between clients |
| Per-client memory limit | `CUDA_MPS_PINNED_DEVICE_MEM_LIMIT` caps device memory | One client can't exhaust GPU memory for others |
| Volta+ memory isolation | Each client gets its own GPU virtual address space | A stray pointer in one client can't corrupt another |
| Volta+ error containment | A fatal GPU fault is scoped to the offending client | One crashing client no longer kills every client |
| Higher client count | Up to 48 concurrent clients per GPU on Volta+ (16 pre-Volta) | Pack many processes onto one GPU |

### Volta MPS vs. pre-Volta MPS

The Volta architecture (2017) rewrote MPS to integrate with hardware: clients submit
work directly to the GPU without passing through the CPU-side server for every launch,
each client gets isolated address spaces, and faults are contained. Pre-Volta MPS
shares a single address space with no isolation and routes all launches through the
server, so treat the two as meaningfully different in isolation guarantees.

## Architecture Overview

MPS has three pieces: a **control daemon**, a **server**, and the **client runtime**
linked into every CUDA application.

```
   Client A (CUDA app)   Client B (CUDA app)   Client C (CUDA app)
          |                    |                     |
          |   named pipe ($CUDA_MPS_PIPE_DIRECTORY)  |
          +----------+---------+----------+----------+
                     |                    |
              nvidia-cuda-mps-control  (the daemon, started with -d)
                     |  starts / manages
                     v
              nvidia-cuda-mps-server   (one per user, owns ONE CUDA context)
                     |
                     v
        +-----------------------------------+
        |   GPU  (SMs shared concurrently)  |
        +-----------------------------------+
```

### Components

1. **Control daemon — `nvidia-cuda-mps-control`**: started once per node (with `-d`).
   It launches/stops server processes, routes new clients to the right server, and
   accepts interactive control commands (listing clients, setting thread percentages,
   memory limits).
2. **Server — `nvidia-cuda-mps-server`**: owns the single shared CUDA context and
   issues the merged stream of work to the GPU. There is **one server per user** per
   GPU; the daemon spawns it on demand when the first client of a user connects.
3. **Client runtime**: the standard CUDA driver inside each application detects the
   pipe directory and transparently connects to the server instead of creating its own
   GPU context. No relinking or code change is needed.

Two directories tie it together, both overridable by environment variable:
`CUDA_MPS_PIPE_DIRECTORY` (default `/tmp/nvidia-mps`, the control socket) and
`CUDA_MPS_LOG_DIRECTORY` (default `/var/log/nvidia-mps`, server/control logs).

## Installation

MPS ships **with the NVIDIA CUDA driver** — there is nothing to `pip install`. The
control and server binaries live alongside the driver (typically `/usr/bin`).

### Prerequisites

- An NVIDIA GPU and a matching NVIDIA driver (Volta or newer strongly recommended for
  isolation and per-client provisioning).
- CUDA toolkit only if you intend to *build* CUDA apps; the MPS binaries themselves
  come from the driver package.
- Root (or the relevant user) to set the GPU compute mode and start the daemon.

### Verify the binaries are present

In [ ]:
# These tools are part of the NVIDIA driver install — verify they exist.
# (Run in a shell on a GPU host; shown here as the commands you'd use.)
import shutil

for tool in ("nvidia-smi", "nvidia-cuda-mps-control", "nvidia-cuda-mps-server"):
    path = shutil.which(tool)
    print(f"{tool:28} -> {path or 'NOT FOUND (install/locate the NVIDIA driver)'}")

## Basic Usage

The classic single-user workflow: optionally put the GPU in `EXCLUSIVE_PROCESS`
compute mode (so the daemon owns it), start the daemon, run your jobs, then shut it
down. All four steps are shell commands run on the GPU host.

```bash
# 1. (Optional) restrict the daemon to GPU 0 and set exclusive compute mode so
#    only the MPS server can create a context on it.
export CUDA_VISIBLE_DEVICES=0
sudo nvidia-smi -i 0 -c EXCLUSIVE_PROCESS      # compute mode 3

# 2. Point MPS at writable pipe/log dirs and start the control daemon (-d = detach).
export CUDA_MPS_PIPE_DIRECTORY=/tmp/nvidia-mps
export CUDA_MPS_LOG_DIRECTORY=/tmp/nvidia-mps-log
nvidia-cuda-mps-control -d

# 3. Launch your CUDA applications normally — they auto-connect to the MPS server.
./my_inference_worker &
./my_inference_worker &
mpirun -np 4 ./my_hpc_app

# 4. Shut the daemon down cleanly and restore the default compute mode.
echo quit | nvidia-cuda-mps-control
sudo nvidia-smi -i 0 -c DEFAULT                # compute mode 0
```

### Interactive control commands

The daemon accepts commands on stdin. Pipe a single command, or run it interactively:

```bash
echo "get_server_list"              | nvidia-cuda-mps-control
echo "get_device_client_list"       | nvidia-cuda-mps-control
```

### Helper: start/stop the daemon from Python

In [ ]:
import os
import subprocess


def start_mps(pipe_dir="/tmp/nvidia-mps", log_dir="/tmp/nvidia-mps-log"):
    """Start the MPS control daemon, returning the environment clients must inherit."""
    env = dict(os.environ, CUDA_MPS_PIPE_DIRECTORY=pipe_dir,
               CUDA_MPS_LOG_DIRECTORY=log_dir)
    os.makedirs(pipe_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    subprocess.run(["nvidia-cuda-mps-control", "-d"], env=env, check=True)
    return env  # pass this env to every CUDA client you launch


def mps_control(command, env):
    """Send one control command (e.g. 'get_server_list') to the running daemon."""
    return subprocess.run(["nvidia-cuda-mps-control"], input=command + "\n",
                          env=env, text=True, capture_output=True).stdout


def stop_mps(env):
    """Shut the daemon (and its servers) down cleanly."""
    subprocess.run(["nvidia-cuda-mps-control"], input="quit\n", env=env, text=True)


# Example (requires a GPU host with the NVIDIA driver):
#   env = start_mps()
#   print(mps_control("get_server_list", env))
#   ... launch CUDA workers with env ...
#   stop_mps(env)
print("Defined start_mps / mps_control / stop_mps helpers.")

## Advanced Features

### Provisioning SMs per client (Volta+)

By default every client may use 100% of the GPU's SMs and they contend for them. You
can cap the **active thread percentage** to give each client a slice of the SMs — a
soft spatial partition that improves predictability and prevents one greedy client
from starving the rest.

Two ways to set it:

- **Per launch**, via the client's environment:
  `CUDA_MPS_ACTIVE_THREAD_PERCENTAGE=25 ./worker` limits that worker to ~25% of SMs.
- **Globally / per active client**, via control commands:
  `set_default_active_thread_percentage 50` or
  `set_active_thread_percentage <server-PID> 50`.

### Capping device memory per client (Volta+)

`CUDA_MPS_PINNED_DEVICE_MEM_LIMIT` (or the control command
`set_default_device_pinned_mem_limit <dev> <size>`) bounds how much device memory a
single client can pin, so one client can't OOM the others:

```bash
# Give each client at most 25% of SMs and 4 GB of device memory.
export CUDA_MPS_ACTIVE_THREAD_PERCENTAGE=25
export CUDA_MPS_PINNED_DEVICE_MEM_LIMIT="0=4G"
./worker
```

In [ ]:
# Build the environment for a single throttled MPS client.
def client_env(base_env, active_thread_pct=None, mem_limit=None, device=0):
    """Return env vars that cap an MPS client's SM share and device memory."""
    env = dict(base_env)
    if active_thread_pct is not None:
        env["CUDA_MPS_ACTIVE_THREAD_PERCENTAGE"] = str(active_thread_pct)
    if mem_limit is not None:
        # Format expected by the driver, e.g. "0=4G" -> 4 GB cap on device 0.
        env["CUDA_MPS_PINNED_DEVICE_MEM_LIMIT"] = f"{device}={mem_limit}"
    return env


demo = client_env({"PATH": "/usr/bin"}, active_thread_pct=25, mem_limit="4G")
for k in ("CUDA_MPS_ACTIVE_THREAD_PERCENTAGE", "CUDA_MPS_PINNED_DEVICE_MEM_LIMIT"):
    print(f"{k} = {demo[k]}")

## Use Cases

### Real-world Applications of MPS

#### Use Case 1: Multi-model inference serving

- **Context**: A GPU hosting several small models (or replicas of one model) where any
  single request only lightly loads the GPU.
- **Implementation**: Run each model worker as its own process under one MPS daemon;
  optionally give latency-critical workers a larger `ACTIVE_THREAD_PERCENTAGE`.
- **Results**: Concurrent kernel execution raises aggregate throughput and GPU
  utilization versus serial time-slicing, often with lower tail latency. (NVIDIA
  Triton can manage MPS-style concurrency directly.)

#### Use Case 2: HPC / MPI ranks sharing a GPU

- **Context**: A legacy MPI application that assigns one rank per GPU but each rank
  only uses a fraction of the device.
- **Implementation**: Oversubscribe — launch several ranks per GPU under MPS
  (`mpirun -np N`), letting their kernels overlap in the shared context.
- **Results**: Better strong-scaling and device utilization without rewriting the app.

#### Use Case 3: Packing batch jobs onto idle GPUs

- **Context**: A scheduler with many short, GPU-light jobs from the same trusted user.
- **Implementation**: Co-locate jobs on one GPU under MPS with per-client SM/memory
  caps to bound interference.
- **Results**: Higher cluster throughput and fewer stranded GPU cycles.

## Best Practices

1. **Match GPU compute mode to your model.** For a single-user node, set
   `EXCLUSIVE_PROCESS` so only the MPS server creates a context. For shared nodes leave
   it `DEFAULT` and let MPS coexist with non-MPS jobs.
2. **Run one daemon per user.** A server services exactly one user's clients; multiple
   users need separate daemons with distinct `CUDA_MPS_PIPE_DIRECTORY` values.
3. **Use writable, per-node pipe/log directories.** Set `CUDA_MPS_PIPE_DIRECTORY` and
   `CUDA_MPS_LOG_DIRECTORY` explicitly (especially in containers) to avoid permission
   clashes in `/tmp` and `/var/log`.
4. **Provision SMs deliberately.** Set `CUDA_MPS_ACTIVE_THREAD_PERCENTAGE` so the sum
   across clients reflects your QoS goals instead of relying on free-for-all contention.
5. **Cap per-client memory.** Use `CUDA_MPS_PINNED_DEVICE_MEM_LIMIT` so one client
   can't OOM the GPU for its neighbors.
6. **Prefer Volta or newer.** You get per-client address-space isolation and error
   containment that older GPUs lack.
7. **Shut down cleanly.** `echo quit | nvidia-cuda-mps-control` before host
   maintenance so clients and the server exit gracefully and pipes are cleaned up.

## Common Pitfalls

1. **Treating MPS as a security boundary.** MPS gives memory *isolation* (Volta+) but
   not the strong hardware/fault/bandwidth isolation of MIG. Don't use it to separate
   untrusted tenants — clients still share one context and one fault domain.
2. **Mixing users on one daemon.** A single server serves one user only. Jobs from a
   second user won't attach; they need their own daemon and pipe directory.
3. **Wrong compute mode.** Setting `EXCLUSIVE_PROCESS` and then launching non-MPS apps
   makes them fail to get a context; conversely, forgetting it on a single-user node
   lets stray processes create their own contexts outside MPS.
4. **Stale pipe/log directories.** A crashed daemon can leave sockets behind; new
   clients then hang or fail to connect. Clear `CUDA_MPS_PIPE_DIRECTORY` and restart.
5. **Over-subscribing memory.** Concurrency multiplies memory pressure; without
   per-client limits the combined footprint OOMs the GPU.
6. **Expecting MPS inside MIG to span instances.** MPS runs *within* a single GPU or a
   single MIG instance — it does not let clients share across MIG partitions.

## Performance Optimization

### Tuning MPS for throughput and QoS

#### Configuration tuning

- **`CUDA_MPS_ACTIVE_THREAD_PERCENTAGE`** — the SM slice per client. Smaller slices
  isolate clients but cap any single client's peak; size it to the number of co-located
  clients (e.g. ~`100 / N`% as a starting point) and adjust by measurement.
- **`CUDA_MPS_PINNED_DEVICE_MEM_LIMIT`** — device memory cap per client. Set so
  `N × limit` fits comfortably in GPU memory with headroom for fragmentation.
- **Client count** — concurrency helps until SM, memory-bandwidth, or memory-capacity
  contention dominates; benchmark to find the knee rather than maximizing client count.
- **Kernel size** — MPS shines for *small* kernels that individually under-fill the GPU;
  workloads that already saturate the SMs gain little and may regress under contention.

#### How to measure the win

Compare aggregate throughput (samples/s or requests/s) **with** and **without** MPS at
your real concurrency, and watch `nvidia-smi` GPU utilization. Below the helper sketches
a simple A/B harness.

In [ ]:
import subprocess
import time


def run_workers(launch_cmd, n_workers, env=None):
    """Launch n identical worker processes, wait for all, return wall-clock seconds.

    Run this twice — once with the plain environment and once with the MPS env from
    start_mps() — to quantify the throughput gain from concurrent GPU sharing.
    """
    start = time.perf_counter()
    procs = [subprocess.Popen(launch_cmd, env=env) for _ in range(n_workers)]
    for p in procs:
        p.wait()
    return time.perf_counter() - start


# Example (needs a real GPU workload binary):
#   baseline = run_workers(["./infer", "--reqs", "1000"], 4)            # time-sliced
#   with_mps = run_workers(["./infer", "--reqs", "1000"], 4, env=env)   # MPS shared
#   print(f"speedup: {baseline / with_mps:.2f}x")
print("Defined run_workers() A/B harness for MPS-on vs MPS-off comparison.")

## Production Deployment

### Running MPS in containers and Kubernetes

#### Docker

A container needs the NVIDIA Container Toolkit, access to the GPU, and a *shared* pipe
directory so the daemon and client processes see the same sockets:

```dockerfile
FROM nvidia/cuda:12.4.0-runtime-ubuntu22.04

ENV CUDA_MPS_PIPE_DIRECTORY=/tmp/nvidia-mps \
    CUDA_MPS_LOG_DIRECTORY=/tmp/nvidia-mps-log

# Start the daemon, then exec the workload (use a real entrypoint script in practice).
CMD nvidia-cuda-mps-control -d && \
    ./inference_server
```

Run with the GPU exposed:

```bash
docker run --gpus all --ipc=host my-mps-image
```

#### Kubernetes

Don't hand-roll the daemon in Kubernetes — use the **NVIDIA GPU Operator / k8s-device-plugin**,
which supports an `mps` sharing strategy. The plugin starts and manages the MPS control
daemon and advertises *replicas* of each physical GPU so multiple pods land on one device:

```yaml
# ConfigMap consumed by the NVIDIA device plugin (GPU Operator).
apiVersion: v1
kind: ConfigMap
metadata:
  name: device-plugin-config
  namespace: gpu-operator
data:
  mps-config: |-
    version: v1
    sharing:
      mps:
        resources:
          - name: nvidia.com/gpu
            replicas: 4      # advertise each GPU as 4 MPS-shared units
```

A pod then simply requests one shared unit and the plugin routes it to an MPS server:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: mps-inference
spec:
  containers:
    - name: worker
      image: my-mps-image
      resources:
        limits:
          nvidia.com/gpu: 1   # one MPS replica, not a whole physical GPU
```

## Monitoring and Observability

### Watching MPS in production

#### Key metrics to track

- **GPU utilization & SM activity** — the headline signal that sharing is filling the
  device. From `nvidia-smi dmon` or DCGM (`DCGM_FI_DEV_GPU_UTIL`, `DCGM_FI_PROF_SM_ACTIVE`).
- **Per-process memory & compute** — `nvidia-smi` shows clients, but under MPS their
  work appears under the *server* PID; correlate with the daemon's client list.
- **Active client count** — from `get_device_client_list`; compare against your expected
  concurrency and the 48-client ceiling.
- **Per-client throughput / latency** — application-level (requests/s, p99 latency) to
  confirm sharing isn't degrading QoS for any client.

#### Tooling

```bash
nvidia-smi dmon -s u                 # live SM/memory utilization, 1 Hz
echo "get_device_client_list" | nvidia-cuda-mps-control   # who is attached
dcgmi dmon -e 1002,1003,203          # SM active, SM occupancy, GPU util via DCGM
```

#### Logging

- The server and control daemon write to `CUDA_MPS_LOG_DIRECTORY` (`server.log`,
  `control.log`) — ship these to your log stack.
- Set log verbosity with control commands (`set_default_logging_level`) when debugging.
- Emit structured app logs tagged with the client identity so per-client behavior is
  attributable even though kernels run under the shared server context.

## Troubleshooting

#### Issue 1: Clients hang or fail to start a context

**Symptoms**: A CUDA app blocks on init, or returns `CUDA_ERROR_NOT_PERMITTED` /
"all CUDA-capable devices are busy or unavailable".

**Cause**: GPU is in `EXCLUSIVE_PROCESS` mode and the MPS server already holds the only
allowed context, or a stale daemon left dead pipes in `CUDA_MPS_PIPE_DIRECTORY`.

**Solution**: Confirm the daemon is up (`get_server_list`); ensure clients export the
same `CUDA_MPS_PIPE_DIRECTORY`; if stale, `echo quit | nvidia-cuda-mps-control`, delete
the pipe directory, and restart the daemon.

#### Issue 2: A second user's jobs won't attach

**Symptoms**: Process from user B can't connect to the running MPS server.

**Cause**: An MPS server serves exactly one user; B has no server.

**Solution**: Start a separate daemon for user B with its own `CUDA_MPS_PIPE_DIRECTORY`
(and matching compute-mode arrangement).

#### Issue 3: One client crash takes down the others (pre-Volta)

**Symptoms**: A fault in one client makes sibling clients fail.

**Cause**: Pre-Volta MPS has no error containment — all clients share one fault domain.

**Solution**: Move to Volta or newer GPUs (per-client isolation + containment), or fall
back to time-slicing / MIG where fault isolation is required.

#### Issue 4: Out-of-memory under concurrency

**Symptoms**: `CUDA_ERROR_OUT_OF_MEMORY` only when several clients run together.

**Cause**: Combined footprint of concurrent clients exceeds GPU memory.

**Solution**: Set `CUDA_MPS_PINNED_DEVICE_MEM_LIMIT` per client and/or reduce client
count so `N × limit` fits with headroom.

## Comparison with Alternatives

### How MPS compares to other GPU-sharing approaches

| Aspect | MPS | Time-slicing (default) | MIG (Multi-Instance GPU) |
|--------|-----|------------------------|--------------------------|
| Sharing model | Spatial — concurrent kernels in one context | Temporal — serial context swaps | Hardware partitions (separate SMs/memory/cache) |
| Isolation | Memory isolation (Volta+); shared fault/bandwidth | None | Strong hardware isolation incl. memory bandwidth & faults |
| Granularity | Per-client SM % and memory caps | None (round-robin) | Fixed instance profiles (e.g. 1g.10gb) |
| Reconfiguration | Dynamic, no GPU reset | N/A | Requires partition reconfig (GPU reset) |
| Best for | Many small same-user jobs, inference, MPI | Simple oversubscription, dev | Multi-tenant, guaranteed QoS/SLAs |
| Overhead | Low; one shared context | Context-switch cost | None at runtime; coarse partitioning |
| Hardware | Most NVIDIA GPUs (best on Volta+) | All | A100/H100-class and newer only |

MPS and MIG are complementary: you can run an MPS daemon *inside* a single MIG instance
to share that partition among several processes.

### When to choose MPS

- You have **many small, trusted, same-user** processes that individually under-use the GPU.
- You want **higher utilization without hardware repartitioning** or a GPU reset.
- You **don't need** the strong tenant isolation/QoS guarantees of MIG, and your GPUs
  may predate MIG support.

## Resources

### Official documentation

- NVIDIA MPS documentation: https://docs.nvidia.com/deploy/mps/index.html
- CUDA C++ Programming Guide: https://docs.nvidia.com/cuda/cuda-c-programming-guide/
- `nvidia-smi` compute-mode reference: https://docs.nvidia.com/deploy/nvidia-smi/index.html

### Guides and tutorials

- NVIDIA GPU Operator — time-slicing & MPS sharing:
  https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/gpu-sharing.html
- Kubernetes device plugin (sharing strategies):
  https://github.com/NVIDIA/k8s-device-plugin
- Triton Inference Server — concurrent model execution:
  https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/

### Community and related

- NVIDIA Developer Forums (CUDA / MPS): https://forums.developer.nvidia.com/
- Stack Overflow tag: https://stackoverflow.com/questions/tagged/nvidia-mps

### Related technologies

- **Multi-Instance GPU (MIG)** — hardware partitioning for strong isolation.
- **Time-slicing** — the default temporal sharing MPS replaces.
- **DCGM** — data-center GPU telemetry for monitoring shared GPUs.
- **NVIDIA Container Toolkit / GPU Operator** — exposing and sharing GPUs in containers and k8s.